In [2]:
# ───────────────────────────────────────────────────────────
# (A) TRAIN your models up front, and keep them in a dict:
# ───────────────────────────────────────────────────────────
from pathlib import Path
import polars as pl              # fast DataFrame engine
import pandas as pd # for scoring
import numpy as np
from pathlib import Path
import lightgbm as lgb
from   scipy.stats import rankdata   # for row-wise ranking
import pyarrow as pa

# =========================================================
# 2) Read data with Polars
# =========================================================
#DATA_DIR = Path("/kaggle/input/mitsui-commodity-prediction-challenge")
# local path
DATA_DIR = Path("/home/ubuntu/repos/kaggle-mitsui/data")

# ---------- load ----------
feat_pl   = pl.read_csv(DATA_DIR / "train.csv")          # features   (has date_id)
label_pl  = pl.read_csv(DATA_DIR / "train_labels.csv")   # targets    (has date_id)
test_pl   = pl.read_csv(DATA_DIR / "test.csv")           # test feats (has date_id)

# ---------- join on `date_id` ----------
train_pl = feat_pl.join(label_pl, on="date_id", how="inner")

# ---------- column sets ----------
target_cols  = [c for c in train_pl.columns if c.startswith("target_")]
feature_cols = [c for c in train_pl.columns
                if c not in target_cols + ["date_id"]]


# fill all feature nulls with 0.0  (or median/mean per column)
train_pl = train_pl.with_columns(
    [pl.col(c).fill_null(0.0) for c in feature_cols]
)
test_pl  = test_pl.with_columns(
    [pl.col(c).fill_null(0.0) for c in feature_cols]
)

# ---------- numpy matrices for LightGBM ----------
X_train = train_pl.select(feature_cols).to_numpy()
y_dict  = {t: train_pl.select(t).to_numpy().ravel().astype(float) 
           for t in target_cols}

X_test  = test_pl.select(feature_cols).to_numpy()

# check the data
print("train_pl shape:", train_pl.shape)      # rows, cols
print("X_train shape:", X_train.shape, X_train.dtype)  # (n_rows, n_features)
print("Example target 0 shape:", y_dict[target_cols[0]].shape)



assert not np.isnan(X_train).any(),  "NaNs in features!"
assert X_train.dtype in (np.float32, np.float64), "LightGBM wants float"


# ---------- LightGBM params ----------
params = dict(
    objective        = "regression",
    boosting_type    = "gbdt",
    n_estimators     = 4000,
    learning_rate    = 0.02,
    num_leaves       = 127,
    feature_fraction = 0.8,
    subsample        = 0.7,
    lambda_l2        = 2e-3,
    random_state     = 42,
    verbose          = -1,
    n_jobs           = 16,            # 2 vCPU on Kaggle
)

# uncomment below to use GPU training
#params.update({
#    "device_type":     "gpu",
#    "max_bin":         255,
#    "gpu_platform_id": 0,
#    "gpu_device_id":   0,
#})

# ---------- train one model per target ----------
#  the built-in callback (LightGBM ≥4.0)
stop_cb = lgb.early_stopping(
    stopping_rounds=200,
    first_metric_only=False,   # or True if you monitor the first metric only
    verbose=False,
)

# train LightGBM models per target, store in `models: dict[str, Booster]`
models = {}
split = int(0.8 * X_train.shape[0])         # 80 % → train, 20 % → val
train_idx = slice(0, split)                 # rows 0 … split-1
val_idx   = slice(split, None)              # rows split … end


for k, tgt in enumerate(target_cols, 1):
    dtrain = lgb.Dataset(X_train[train_idx], label=y_dict[tgt][train_idx])
    dval   = lgb.Dataset(X_train[val_idx],   label=y_dict[tgt][val_idx])
    models[tgt] = lgb.train(
        params,
        dtrain,
        valid_sets=[dval],
        callbacks=[stop_cb],
    )
    if k % 25 == 0:          # progress print every 25 models
        print(f"{k}/{len(target_cols)}  best_iter={models[tgt].best_iteration:4}")


print("debugging")
# ---------- predict ----------
pred_dict = {"date_id": test_pl["date_id"]}
for tgt in target_cols:
    pred_col                       = tgt.replace("target_", "prediction_")
    pred_dict[pred_col]            = models[tgt].predict(X_test)

submission_pl = pl.DataFrame(pred_dict)
submission_pl.write_csv("lightgbm_polars_submission.csv")



# ───────────────────────────────────────────────────────────
# (B) Define the `predict(...)` function the server will call
# ───────────────────────────────────────────────────────────
import pandas as pd

NUM_TARGET_COLUMNS = len(target_cols)  # should be 424

def predict(
    test: pl.DataFrame,
    label_lags_1_batch: pl.DataFrame,
    label_lags_2_batch: pl.DataFrame,
    label_lags_3_batch: pl.DataFrame,
    label_lags_4_batch: pl.DataFrame,
) -> pl.DataFrame:
    # test comes in as a single‐row batch
    X = test.select(feature_cols).to_numpy()
    preds = {
        f"target_{i}": models[f"target_{i}"].predict(X).item()
        for i in range(NUM_TARGET_COLUMNS)
    }
    # return a Polars DataFrame with exactly one row
    return pl.DataFrame(preds)

# ───────────────────────────────────────────────────────────
# (C) Wire up & launch the inference server
# ───────────────────────────────────────────────────────────
import os
from kaggle_evaluation.mitsui_inference_server import MitsuiInferenceServer

# hook up your predict()
inference_server = MitsuiInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    # on Kaggle’s hidden test set, they call .serve()
    inference_server.serve()
else:
    # for your local dev, point to the folder with the CSVs
    inference_server.run_local_gateway(
        ("/home/ubuntu/repos/kaggle-mitsui/data",)
    )


train_pl shape: (1917, 982)
X_train shape: (1917, 557) float64
Example target 0 shape: (1917,)
25/424  best_iter=  30
50/424  best_iter=   9
75/424  best_iter=   9
100/424  best_iter=  20
125/424  best_iter=   1
150/424  best_iter=  11
175/424  best_iter=   1
200/424  best_iter=  16
225/424  best_iter=   1
250/424  best_iter=   5
275/424  best_iter=   5
300/424  best_iter=  16
325/424  best_iter=   1
350/424  best_iter=   6
375/424  best_iter=   1
400/424  best_iter=  13
debugging


ModuleNotFoundError: No module named 'kaggle_evaluation'